In [0]:
%pip install scikit-learn mlflow matplotlib pandas numpy
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from mlflow.models.signature import infer_signature
import warnings
warnings.filterwarnings('ignore')

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(
    "/Users/vvce22cseaiml0106@vvce.ac.in/llm_quality_anomaly_detection"
)

print("Setup complete!")

In [0]:
# Load quality score data
df = spark.table("llm_pulse_dev.gold.quality_score_daily").toPandas()
df['feedback_date'] = pd.to_datetime(df['feedback_date'])
df = df.sort_values(['model', 'feedback_date']).reset_index(drop=True)

print(f"Total rows: {len(df)}")
print(f"Date range: {df['feedback_date'].min()} to {df['feedback_date'].max()}")
print(f"Models: {df['model'].unique()}")
print(f"\nQuality score stats per model:")
df.groupby('model')['quality_score_pct'].agg(
    ['mean','std','min','max']
).round(2)

In [0]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

models = df['model'].unique()
colors = {'gpt-4o': '#2196F3', 'claude-3-sonnet': '#9C27B0', 'gpt-3.5-turbo': '#FF9800'}

for i, model in enumerate(models):
    model_df = df[df['model'] == model].copy()
    ax = axes[i]

    # Plot daily quality score
    ax.plot(model_df['feedback_date'],
            model_df['quality_score_pct'],
            color=colors.get(model, 'gray'),
            alpha=0.5, linewidth=1, label='Daily score')

    # Plot 7-day rolling average
    ax.plot(model_df['feedback_date'],
            model_df['rolling_7d_quality_score'],
            color=colors.get(model, 'gray'),
            linewidth=2, label='7-day rolling avg')

    # Draw threshold line — 2 std devs below mean
    mean_score = model_df['quality_score_pct'].mean()
    std_score  = model_df['quality_score_pct'].std()
    threshold  = mean_score - (2 * std_score)

    ax.axhline(threshold, color='red', linestyle='--',
               linewidth=1, alpha=0.7, label=f'Anomaly threshold ({threshold:.1f}%)')
    ax.axhline(mean_score, color='green', linestyle='--',
               linewidth=1, alpha=0.5, label=f'Mean ({mean_score:.1f}%)')

    ax.set_title(f'{model} — quality score over time')
    ax.set_ylabel('Quality score %')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 110)

plt.tight_layout()
plt.show()
print("Any score below the red line = anomaly candidate")

In [0]:
def create_anomaly_features(df):
    """
    Features for anomaly detection.
    We capture how unusual today's quality score is
    compared to recent history.
    """
    df = df.copy()

    for model in df['model'].unique():
        mask = df['model'] == model
        model_data = df[mask]['quality_score_pct']

        # How far is today from the rolling average?
        df.loc[mask, 'deviation_from_7d_avg'] = (
            model_data - df.loc[mask, 'rolling_7d_quality_score']
        )

        # Z-score — how many standard deviations from the mean?
        mean = model_data.mean()
        std  = model_data.std()
        df.loc[mask, 'z_score'] = (model_data - mean) / (std + 1e-8)

        # Rolling std — is quality becoming more volatile?
        df.loc[mask, 'rolling_std_7d'] = (
            model_data.shift(1).rolling(7, min_periods=2).std().fillna(0)
        )

        # Rate of change — did quality drop sharply vs yesterday?
        df.loc[mask, 'day_over_day_change'] = model_data.diff().fillna(0)

        # Lag features
        df.loc[mask, 'quality_lag_1d'] = model_data.shift(1).fillna(mean)
        df.loc[mask, 'quality_lag_7d'] = model_data.shift(7).fillna(mean)

    # Date features
    df['day_of_week'] = df['feedback_date'].dt.dayofweek
    df['month']       = df['feedback_date'].dt.month

    # Model encoding
    model_map = {'gpt-4o': 0, 'claude-3-sonnet': 1, 'gpt-3.5-turbo': 2}
    df['model_encoded'] = df['model'].map(model_map)

    return df.dropna()

df_features = create_anomaly_features(df)

feature_cols = [
    'quality_score_pct',
    'rolling_7d_quality_score',
    'deviation_from_7d_avg',
    'z_score',
    'rolling_std_7d',
    'day_over_day_change',
    'quality_lag_1d',
    'quality_lag_7d',
    'day_of_week',
    'month',
    'model_encoded',
]

print(f"Features ready: {len(df_features)} rows, {len(feature_cols)} features")
df_features[feature_cols].describe().round(3)

In [0]:
X = df_features[feature_cols].values

# Scale features — Isolation Forest works better on scaled data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

with mlflow.start_run(run_name="isolation_forest_quality_v1"):

    # ── Model parameters ───────────────────────────────────
    params = {
        'contamination': 0.05,  # expect ~5% of days to be anomalies
        'n_estimators':  200,
        'max_samples':   'auto',
        'random_state':  42,
    }
    mlflow.log_params(params)
    mlflow.log_param("n_features",  len(feature_cols))
    mlflow.log_param("n_samples",   len(X_scaled))
    mlflow.log_param("features",    feature_cols)

    # ── Train Isolation Forest ─────────────────────────────
    iso_forest = IsolationForest(**params)
    iso_forest.fit(X_scaled)

    # Predictions: -1 = anomaly, 1 = normal
    predictions     = iso_forest.predict(X_scaled)
    anomaly_scores  = iso_forest.score_samples(X_scaled)

    # Convert to readable labels
    df_features = df_features.copy()
    df_features['if_anomaly']       = (predictions == -1).astype(int)
    df_features['if_anomaly_score'] = anomaly_scores  # more negative = more anomalous

    # ── Statistical baseline — 2 standard deviations ──────
    # Classic approach: flag anything below mean - 2*std
    for model in df_features['model'].unique():
        mask = df_features['model'] == model
        mean = df_features.loc[mask, 'quality_score_pct'].mean()
        std  = df_features.loc[mask, 'quality_score_pct'].std()
        threshold = mean - (2 * std)
        df_features.loc[mask, 'stat_threshold']   = threshold
        df_features.loc[mask, 'stat_anomaly']     = (
            df_features.loc[mask, 'quality_score_pct'] < threshold
        ).astype(int)

    # ── Combined alert — flag if EITHER method detects anomaly
    df_features['is_anomaly'] = (
        (df_features['if_anomaly'] == 1) |
        (df_features['stat_anomaly'] == 1)
    ).astype(int)

    # ── Log metrics ────────────────────────────────────────
    total_anomalies = df_features['is_anomaly'].sum()
    anomaly_rate    = df_features['is_anomaly'].mean() * 100

    mlflow.log_metric("total_anomalies_detected", int(total_anomalies))
    mlflow.log_metric("anomaly_rate_pct",          round(anomaly_rate, 2))

    # ── Anomaly visualization ──────────────────────────────
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    models = df_features['model'].unique()
    colors = {'gpt-4o':'#2196F3','claude-3-sonnet':'#9C27B0','gpt-3.5-turbo':'#FF9800'}

    for i, model in enumerate(models):
        m_df = df_features[df_features['model'] == model]
        ax   = axes[i]

        # Normal points
        normal = m_df[m_df['is_anomaly'] == 0]
        ax.scatter(normal['feedback_date'], normal['quality_score_pct'],
                   color=colors.get(model,'gray'), s=15, alpha=0.6, label='Normal')

        # Anomaly points — highlighted in red
        anomalies = m_df[m_df['is_anomaly'] == 1]
        ax.scatter(anomalies['feedback_date'], anomalies['quality_score_pct'],
                   color='red', s=60, zorder=5, marker='x',
                   linewidths=2, label=f'Anomaly ({len(anomalies)} days)')

        ax.plot(m_df['feedback_date'], m_df['rolling_7d_quality_score'],
                color=colors.get(model,'gray'), linewidth=1.5, alpha=0.7)
        ax.axhline(m_df['stat_threshold'].iloc[0], color='red',
                   linestyle='--', alpha=0.5, linewidth=1)

        ax.set_title(f'{model} — anomalies detected in red')
        ax.set_ylabel('Quality %')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 110)

    plt.tight_layout()
    mlflow.log_figure(fig, "anomaly_detection_chart.png")
    plt.show()

    # ── Log model to MLflow ────────────────────────────────
    signature = infer_signature(X_scaled, predictions)
    mlflow.sklearn.log_model(
        iso_forest,
        name = "quality_anomaly_model",
        signature = signature,
        registered_model_name = "llm_pulse_dev.gold.quality_anomaly_model"
    )

    print(f"\nModel trained successfully!")
    print(f"Total anomalies detected: {total_anomalies}")
    print(f"Anomaly rate: {anomaly_rate:.2f}%")
    print(f"\nAnomalies per model:")
    print(df_features.groupby('model')['is_anomaly'].sum())

In [0]:
# Build the alerts table — only keep useful columns
alerts_df = df_features[[
    'feedback_date',
    'model',
    'quality_score_pct',
    'rolling_7d_quality_score',
    'stat_threshold',
    'deviation_from_7d_avg',
    'z_score',
    'if_anomaly',
    'stat_anomaly',
    'is_anomaly',
    'if_anomaly_score',
    'total_feedback',
]].copy()

# Add human readable severity
alerts_df['alert_severity'] = 'normal'
alerts_df.loc[
    (alerts_df['is_anomaly'] == 1) &
    (alerts_df['z_score'] < -2),
    'alert_severity'
] = 'critical'
alerts_df.loc[
    (alerts_df['is_anomaly'] == 1) &
    (alerts_df['z_score'] >= -2),
    'alert_severity'
] = 'warning'

# Save to Delta table in gold schema
(spark.createDataFrame(alerts_df)
     .write
     .mode("overwrite")
     .saveAsTable("llm_pulse_dev.gold.quality_alerts"))

print(f"Saved {len(alerts_df)} rows to gold.quality_alerts")
print(f"\nAlert summary:")
print(alerts_df['alert_severity'].value_counts())

In [0]:
%sql
-- Show all critical and warning alerts
SELECT
    feedback_date,
    model,
    ROUND(quality_score_pct, 2)        AS quality_score,
    ROUND(rolling_7d_quality_score, 2) AS rolling_avg,
    ROUND(z_score, 2)                  AS z_score,
    alert_severity
FROM llm_pulse_dev.gold.quality_alerts
WHERE is_anomaly = 1
ORDER BY feedback_date DESC, z_score ASC
LIMIT 20;

In [0]:
%sql
-- Create unified alert_summary table
-- Combines cost forecast + quality anomaly results

CREATE OR REPLACE TABLE llm_pulse_dev.gold.alert_summary AS

-- Quality alerts
SELECT
    qa.feedback_date          AS alert_date,
    qa.model,
    NULL                      AS team,
    'quality_drop'            AS alert_type,
    qa.alert_severity,
    CONCAT(
        'Quality dropped to ', ROUND(qa.quality_score_pct, 1),
        '% (rolling avg: ', ROUND(qa.rolling_7d_quality_score, 1), '%)'
    )                         AS alert_message,
    qa.quality_score_pct      AS metric_value,
    qa.rolling_7d_quality_score AS baseline_value
FROM llm_pulse_dev.gold.quality_alerts qa
WHERE qa.is_anomaly = 1

UNION ALL

-- Cost spike alerts — flag days where cost > 150% of rolling average
SELECT
    dc.event_date             AS alert_date,
    NULL                      AS model,
    dc.team,
    'cost_spike'              AS alert_type,
    'warning'                 AS alert_severity,
    CONCAT(
        'Daily cost $', ROUND(dc.total_cost_usd, 3),
        ' exceeds 150% of 7-day avg $', ROUND(dc.rolling_7d_avg_cost, 3)
    )                         AS alert_message,
    dc.total_cost_usd         AS metric_value,
    dc.rolling_7d_avg_cost    AS baseline_value
FROM llm_pulse_dev.gold.daily_cost_by_team dc
WHERE dc.total_cost_usd > (dc.rolling_7d_avg_cost * 1.5)
  AND dc.rolling_7d_avg_cost > 0

ORDER BY alert_date DESC;

SELECT COUNT(*) AS total_alerts,
       alert_type,
       alert_severity
FROM llm_pulse_dev.gold.alert_summary
GROUP BY alert_type, alert_severity
ORDER BY total_alerts DESC;